# Micro panel construction

Builds the bank-level panel from the raw ORBIS extract, in one pass:

1. Load the ORBIS export and normalise the column names
2. Reshape wide-by-year to long, then pivot to one row per bank-year
3. Attach the country code
4. Construct the derived ratios and apply the domain bounds
5. Check panel integrity and missingness
6. Rename to the short variable codes and save the preclean panels
7. Build the baseline set (2004-2017, 9% missingness threshold)
8. Build the robustness set (1995-2017, 12% threshold)
9. Convert to ISO3 and save

**Input** `data/micro/raw/orbis_MICRO_all_EU25_m_EUR_1996_2017.xlsx`
**Output** `data/micro/clean/micro_clean_base.parquet`, `micro_clean_robust.parquet`

The ORBIS extract is licensed from Bureau van Dijk and is not distributed with this
repository. See `DATA.md` for the extraction specification needed to reproduce it.

Merged from the original `01-02_micro_dataprep + preclean`, `03_micro_construct_baseline`,
`04_micro_robustness_construction` and `05_micro_convert_isos`. Those four notebooks each
read and rewrote the same parquet files, so running them out of order silently discarded
the ISO conversion. Chaining them in memory here removes that.

## Setup

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pycountry

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# A2/A16/A17 (fixed for the 2026 rerun): non-banks, entities without any balance-sheet data and group
# duplicates are removed in section 4b, before any statistic is computed. See DATA.md, Sample composition.
RAW_XLSX = Path("../data/micro/raw/orbis_MICRO_all_EU25_m_EUR_1996_2017.xlsx")
PRECLEAN_DIR = Path("../data/micro/preclean")
CLEAN_DIR = Path("../data/micro/clean")
DICT_DIR = Path("../data/micro/dictionaries")

for d in (PRECLEAN_DIR, CLEAN_DIR, DICT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Helper functions

Defined once here; the baseline and robustness sets both use `panel_impute`.

In [ ]:
def safe_log(x):
    return np.where(x > 0, np.log(x), np.nan)


def panel_impute(df, id_col="bank_id", time_col="year"): #imputes missing values in the panel. It first sorts by the id and time columns so the time-based operations below work correctly
    df = df.sort_values([id_col, time_col]).copy()
    cols = [c for c in df.columns if c not in [id_col, time_col]] #the list of variables to impute, excluding the identifiers

    df[cols] = (
        df.groupby(id_col)[cols]
          .transform(lambda g: g.interpolate(method="linear", limit_area="inside")) #only fills a gap if there's a real value on both sides of it. Edge NaNs (before the first observation, after the last) stay as NaN
          .reset_index(level=0, drop=True) #for each bank, look at its values over time and fill gaps by drawing a straight line between known values
    )
    df[cols] = df.groupby(id_col)[cols].transform(lambda x: x.fillna(x.mean())) #after interpolation some gaps remain where a bank is missing a long stretch at the start or end. For those, use that specific bank's own average, which is better than the overall mean because banks differ so much
    df[cols] = df.groupby(time_col)[cols].transform(lambda x: x.fillna(x.median())) #anything still missing means the bank has no data at all for that variable, so fall back on what other banks looked like that same year. Median rather than mean because it's more robust to outliers
    return df

## 1. Load the ORBIS extract

In [ ]:
orbisMicroDF = pd.read_excel(RAW_XLSX, sheet_name=1) #sheet 1 holds the financial data, sheet 0 holds the company info
print("Loaded df shape:", orbisMicroDF.shape)
print("Unique banks:", orbisMicroDF["BvD ID number"].nunique())

In [ ]:
orbisMicroDF.columns = orbisMicroDF.columns.astype(str)
orbisMicroDF.columns = orbisMicroDF.columns.str.replace("\nm ", " ", regex=False)
orbisMicroDF.columns = orbisMicroDF.columns.str.replace(r"\s+", " ", regex=True).str.strip()
orbisMicroDF = orbisMicroDF.rename(columns={'BvD ID number': 'bank_id'})
orbisMicroDF = orbisMicroDF.drop(columns=["Unnamed: 0"], errors = "ignore") # Drop the "Unnamed: 0" column if it exists, without raising an error if it doesn't

In [ ]:
missing_codes = ["n.a.", "n.s.", "n.d.", "na", "NA", "N/A", ""]
orbisMicroDF = orbisMicroDF.replace(missing_codes, np.nan)

## 2. Wide to long

The export is one row per bank with a column per variable-year, so the year has to be
split off the column name before it can be pivoted into a panel.

In [ ]:
year_pattern = re.compile(r"(19|20)\d{2}$")
year_cols = [c for c in orbisMicroDF.columns if year_pattern.search(c)]
id_cols = [c for c in orbisMicroDF.columns if c not in year_cols]

print("Found year columns:", len(year_cols))
print("Example year columns:", year_cols[:8])

if len(year_cols) == 0: #fallback for a less strict match, in case the export stores the year somewhere other than the end of the column name
    year_cols = [c for c in orbisMicroDF.columns if re.search(r"(19|20)\d{2}", c)]
    id_cols = [c for c in orbisMicroDF.columns if c not in year_cols]
    print("Fallback year columns found:", len(year_cols))

    if len(year_cols) == 0:
        raise ValueError(
            "No year columns found. The export may not be wide-by-year. "
            "Check orbisMicroDF.columns and confirm how years are stored."
        )

In [ ]:
long = orbisMicroDF.melt( #Each row is one bank x variable x year combination
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="var_year", #This will contain the original column names, which include the variable name and year (e.g., "Total Assets 2016")
    value_name="value"
)

long["year"] = long["var_year"].str.extract(r"((?:19|20)\d{2})").astype(int) #Extract the year from the original column name and convert to int

long["variable"] = ( #Extract the variable name by removing the year from the original column name
    long["var_year"]
    .str.replace(r"\s*((?:19|20)\d{2})\s*$","", regex=True)
    .str.strip()
)

long = long.drop(columns=["var_year"])

print("\nLong shape:", long.shape)
print("Year range:", long["year"].min(), "-", long["year"].max())
print("Unique variables:", long["variable"].nunique())
print(long[["variable", "year"]].sample(10))

## 3. Long to bank-year panel

In [ ]:
bank_id = "bank_id"

panel = long.pivot_table( #Instead of one row per bank-variable-year, we want one row per bank-year, and columns for each variable
    index=[bank_id,"year"],
    columns="variable",
    values="value",
    aggfunc="first" #If there are multiple entries for the same bank-year-variable, take the first one (there shouldn't be any if the data is clean)
).reset_index() #Reset index to turn bank_id and year back into columns

non_numeric = ["bank_id", "year", "country_iso"]
for col in panel.columns:
     if col not in non_numeric:
        panel[col] = pd.to_numeric(panel[col], errors="coerce") #Convert to numeric, set non-convertible values to NaN

panel.columns.name=None

print("\nPanel shape:", panel.shape)
print("Panel columns (first 20):", list(panel.columns[:20]))
display(panel.head())

## 4. Add country ISO code

In [ ]:
bank_country = ( #After the pivot, country info is lost, so we need to get it from the original df. We take the first non-missing country for each bank_id.
    orbisMicroDF[["bank_id", "Country ISO code"]]
    .dropna(subset=["bank_id", "Country ISO code"])
    .drop_duplicates(subset=["bank_id"]) #Ensures each bank appears only once in the lookup, one bank, one country
    .rename(columns={"Country ISO code": "country_iso"}) 
)

panel = panel.merge(bank_country, on="bank_id", how="left") #Keep all rows in panel, and attach country where available, if a bank_id doesn't have a country, it will be NaN

print("Missing country share:", panel["country_iso"].isna().mean())

## 4b. Sample corrections

Three kinds of entity are removed here, before any statistic is computed, so the missingness
shares, the imputation and every downstream panel rest on the corrected sample (audit 2026,
issues A2, A16, A17). The variable selection further down is kept as published; see section 10.

In [ ]:
# A2: the NACE 641 search also returns 6411 (central banking). ORBIS assigns that code to a number
# of Irish holding companies (and two central banks), which are not banks. 6419 is "other monetary
# intermediation", the commercial, savings and cooperative banks the thesis is about.
NACE_COL = "NACE Rev. 2, core code (4 digits)"
entity_info = orbisMicroDF.drop_duplicates("bank_id").set_index("bank_id")
non_banks = set(entity_info.index[entity_info[NACE_COL] != 6419])

# A16: entities for which ORBIS reports none of the five core balance-sheet items in any year. Every
# one of their deposits, loans, equity, interbank and interest-income values would be filled with the
# year median, so the models would see a median bank carrying only their real size and ROA/ROE.
CORE_ITEMS = [
    "Total customer deposits EUR", "Gross loans & advances to customers EUR", "Total equity EUR",
    "Interbank liabilities EUR", "Net interest income (expense) EUR",
]
has_core = panel.groupby("bank_id")[CORE_ITEMS].apply(lambda g: g.notna().to_numpy().any())
no_balance_sheet = set(has_core.index[~has_core])

# A17: parent and subsidiary both in the extract with consolidated (C2) accounts, so the same balance
# sheet is counted twice. The group or foundation entity is dropped and the operating bank kept.
GROUP_DUPLICATES = [
    "ABN AMRO GROUP N.V.", "MONTEPIO GERAL - ASSOCIACAO MUTUALISTA", "BANK OF IRELAND GROUP PLC",
    "AIB GROUP PUBLIC LIMITED COMPANY", "FUNDACION BANCARIA IBERCAJA", "FUNDACION BANCARIA UNICAJA",
]
names = entity_info["Company name Latin alphabet"]
group_duplicates = set(names.index[names.isin(GROUP_DUPLICATES)])
assert set(names[names.isin(GROUP_DUPLICATES)]) == set(GROUP_DUPLICATES), "a listed group entity is missing from the extract"

drop_ids = non_banks | no_balance_sheet | group_duplicates
print(f"Non-banks (NACE != 6419):        {len(non_banks)}")
print(f"No core balance-sheet item ever: {len(no_balance_sheet)} ({len(no_balance_sheet & non_banks)} of them also non-banks)")
print(f"Group duplicates:                {len(group_duplicates)}")
print(f"Entities removed in total:       {len(drop_ids)} of {panel['bank_id'].nunique()}")

rows_before = len(panel)
panel = panel[~panel["bank_id"].isin(drop_ids)].reset_index(drop=True)
print(f"Bank-years: {rows_before} -> {len(panel)}")

## 5. Equity ratio

Constructed first because the domain bounds applied to it decide which rows survive.

In [ ]:
equity_candidates = [c for c in panel.columns if c.lower() == "total equity eur"]
assets_candidates = [c for c in panel.columns if c.lower() == "total assets eur"]

if len(equity_candidates) != 1 or len(assets_candidates) != 1: #stop here rather than silently continuing without the ratio, since everything downstream depends on it
    raise KeyError(
        f"Expected exactly one equity and one assets column, found "
        f"{equity_candidates} and {assets_candidates}"
    )

panel["equity_ratio"] = panel[equity_candidates[0]] / panel[assets_candidates[0]]
print(panel["equity_ratio"].describe())

In [ ]:
panel.loc[panel["equity_ratio"] < 0, "equity_ratio"] = np.nan #Technically insolvent, can happen, but its more likely to be an accounting error or data quality error

panel.loc[panel["equity_ratio"] > 1.5, "equity_ratio"] = np.nan #Mathematically impossible, likely data error, can be due to misreporting or misclassification of variables

panel["equity_ratio"].describe()

In [ ]:
panel = panel.rename(columns={"equity_ratio" : "Equity Ratio"})

## 6. Constructed variables

Ten indicators built from the twenty-one raw ORBIS lines. Outliers are handled by hard
domain bounds that set the value to NaN, not by winsorising, so the remaining tails are
real.

In [ ]:
# Sorteer eerst
panel = panel.sort_values(["bank_id", "year"]).copy() #sort by bank_id and year to ensure correct order for growth calculations

# Helper
def safe_log(x):
    return np.where(x > 0, np.log(x), np.nan)

TA   = "Total assets EUR"
EQ   = "Total equity EUR"
LEQ  = "Total liabilities and equity EUR"
LOAN = "Gross loans & advances to customers EUR"
DEP  = "Total customer deposits EUR"

# 1) log assets
panel["Log Total Assets"] = safe_log(panel[TA])

# 2) Asset growth (Δ log TA) #We log to compress the scale and to make the distribution more symmetric, as well as to interpret growth rates as differences in logs
panel["Asset Growth"] = (
    panel.groupby("bank_id")["Log Total Assets"].diff() #Ensures the difference is always calculated within the same bank, and diff takes the difference between consecutive rows, which after sorting by bank and year means year-on-year changes in log assets, which can be interpreted as growth rates
)

# 3) Loan growth (Δ log loans)
panel["Log Loans"] = safe_log(panel[LOAN]) #safe log of loans to handle zeros and negative assets, as this is essentially impossible. It protects against data errors rather than real economic events
panel["Loan Growth"] = (
    panel.groupby("bank_id")["Log Loans"].diff()
)

# 4) Loans / Assets
panel["Loans To Assets"] = panel[LOAN] / panel[TA]

# 5) Loans / Deposits
panel["Loans To Deposits"] = panel[LOAN] / panel[DEP]

# 6) Deposits / Liabilities
panel["Total Liabilities"] = panel[LEQ] - panel[EQ]
panel["Deposits To Liabilities"] = panel[DEP] / panel["Total Liabilities"]

# 7) Leverage = Assets / Equity
panel["Leverage / Assets to Equity"] = panel[TA] / panel[EQ]

In [ ]:
panel.replace([np.inf, -np.inf], np.nan, inplace=True) #Replace infinite values with NaN, which can occur if there are zeros in the denominators of the ratios, or if there are zero or negative values in the log transformations. This ensures that these cases are treated as missing rather than causing errors in analysis.

In [ ]:
panel.loc[panel["Leverage / Assets to Equity"] < 0, "Leverage / Assets to Equity"] = np.nan 
panel.loc[panel["Leverage / Assets to Equity"] > 100, "Leverage / Assets to Equity"] = np.nan #cap it, EU banks under basel regulations must hold far more capital than that. Anything above probably data error

In [ ]:
panel.loc[panel["Loans To Deposits"] > 5, "Loans To Deposits"] = np.nan #No lower bound, but if a bank has more than 5 euros of loans for every euro of deposits, it's likely a data error, as this would be an extremely unusual funding structure for a bank. This is a more lenient threshold than the leverage ratio, as some banks can have very high loan-to-deposit ratios, especially if they rely heavily on wholesale funding, but above 5 is likely unrealistic and indicative of data issues.

## 7. Integrity checks

Read-only. Nothing below this heading changes the panel.

In [ ]:
# 1) No duplicates before the pivot
dups = long.groupby(["bank_id", "year", "variable"]).size()
print("Max rows per (bank_id, year, variable):", dups.max()) #pivot_table with aggfunc="first" would silently keep one of them, so anything above 1 has to be caught here

# 2) Panel coverage
print("\nBanks:", panel["bank_id"].nunique())
print("Years:", panel["year"].min(), "-", panel["year"].max())
print("Rows:", len(panel))

# 3) How much of the panel survives the 2004-2017 window
panel_04017 = panel[(panel["year"] >= 2004) & (panel["year"] <= 2017)].copy()
print("\nRows in 2004-2017:", len(panel_04017))
print("Banks in 2004-2017:", panel_04017["bank_id"].nunique())

# 4) Years observed per bank, which is what the 2004 start date was chosen on
years_per_bank = pd.DataFrame({
    "Full sample": panel.groupby("bank_id")["year"].nunique().describe(),
    "2004-2017": panel_04017.groupby("bank_id")["year"].nunique().describe(),
})
print("\nYears observed per bank:")
print(years_per_bank.round(2))

# 5) Share of each year's banks that were also present the year before
banks_by_year = panel_04017.groupby("year")["bank_id"].apply(set)
overlap = {
    year: len(banks_by_year[year] & banks_by_year[year - 1]) / len(banks_by_year[year])
    for year in banks_by_year.index if year - 1 in banks_by_year
}
print("\nYear-on-year bank overlap:")
print(pd.Series(overlap).describe().round(3))

In [ ]:
new_vars = [
    "Log Total Assets", #log of total assets, a common transformation to handle skewness and interpret coefficients as elasticities
    "Asset Growth", #year-on-year growth rate of total assets, a key indicator of bank growth and expansion
    "Loan Growth", #growth of loans, a key indicator of bank lending activity and risk-taking
    "Loans To Assets", #what share of the bank's assets are loans, a key indicator of business model and risk profile
    "Loans To Deposits", #how many loans are funded by deposits, a key indicator of liquidity risk and funding structure
    "Deposits To Liabilities", #what share of the bank's liabilities are deposits, a key indicator of funding structure and liquidity risk
    "Leverage / Assets to Equity" #how many euros of assets are funded by each euro of equity, a key indicator of leverage and risk-taking
]

print(panel[new_vars].describe())
print("\nMissingness (%):")
print(panel[new_vars].isna().mean().mul(100).round(2))

### Mergers and restructuring

The panel treats each `bank_id` as one continuous series. This checks where that
assumption breaks, which is what a jump of this size in total assets normally means.

In [ ]:
panel = panel.sort_values(["bank_id", "year"])
ta_growth = panel.groupby("bank_id")["Total assets EUR"].pct_change() #held as a separate series rather than a column, so this check cannot leak into the saved panel

jumps = panel.loc[ta_growth > 2, ["bank_id", "year", "Total assets EUR"]].copy()
jumps["Total Asset Growth"] = ta_growth[ta_growth > 2] #a jump this large is almost always a merger or a restructuring rather than organic growth, so it flags where the bank-year series is not really continuous

print("Bank-years where total assets more than tripled:", len(jumps))
display(jumps.head(20))

### Missingness, full panel against the baseline window

In [ ]:
missing_full = panel.isna().mean().mul(100).round(2).sort_values(ascending=False)
missing_04017 = panel_04017.isna().mean().mul(100).round(2).sort_values(ascending=False)

missing_compare = pd.DataFrame({
    "Full panel": missing_full,
    "2004-2017": missing_04017,
}).reset_index()
missing_compare.columns = ["variable", "Full panel", "2004-2017"]

missing_compare.to_csv(CLEAN_DIR / "missingness_full_comparison.csv", index=False)
missing_compare.head(20)

## 8. Variable dictionary

Maps the ORBIS names to the short codes used everywhere downstream. The prefix marks the
block: `c_` capital, `cr_` credit risk, `p_` profitability, `inc_`/`exp_` income and cost,
`r_` ratios, `l_` logs, `x_` levels, `g_` growth.

In [ ]:
rename_dict = {

# Capital
"Total capital adequacy ratio": "c_capratio",
"Equity Ratio": "c_eqratio",
"Leverage / Assets to Equity": "c_leverage",
"Total equity EUR": "x_equity",

# Credit risk
"Impaired / Non-performing loans to customers EUR": "cr_npl",
"Loan Loss Reserves (Memo) EUR": "cr_llr",
"Net impairment charges on loans & advances EUR": "cr_impchg",

# Profitability
"ROA using Net income": "p_roa",
"ROE using Net income": "p_roe",
"Net interest margin (average total earning assets)": "p_nim",

# Income & cost
"Net interest income (expense) EUR": "inc_nii",
"Operating Income EUR": "inc_op",
"Total operating expenses EUR": "exp_total",
"Cost-to-income (Efficiency) ratio": "r_cti",

# Lending
"Net loans & advances to customers EUR": "x_loans_net",
"Gross loans & advances to customers EUR": "x_loans_gross",
"Log Loans": "l_loans",
"Loans To Assets": "r_loans_assets",
"Loans To Deposits": "r_loans_dep",
"Loan Growth": "g_loans",

# Deposits & funding
"Total customer deposits EUR": "x_dep",
"Deposits To Liabilities": "r_dep_liab",
"Interbank liabilities EUR": "x_ib_liab",
"Interbank assets / Interbank liabilities": "r_ib_ratio",

# Balance sheet size
"Total assets EUR": "x_assets",
"Log Total Assets": "l_assets",
"Total liabilities and equity EUR": "x_liab_equity",
"Total Liabilities": "x_liab",

# Exposure
"Total off-balance sheet exposure EUR": "x_obsi",

# Growth
"Asset Growth": "g_assets",

}

In [ ]:
pd.DataFrame(rename_dict.items(), columns=["original", "short"]).to_csv(
    DICT_DIR / "variable_names_micro.csv", index=False
)
print("Dictionary entries:", len(rename_dict))

## 9. Save the preclean panels

In [ ]:
panel.to_parquet(PRECLEAN_DIR / "micro_fullpanel_preclean.parquet") #saved before the rename, so the full panel keeps the original ORBIS names and the dictionary above stays the single place the mapping is recorded

panel = panel.rename(columns=rename_dict)

#Baseline: 2004-2017
base_preclean = panel[(panel.year >= 2004) & (panel.year <= 2017)].copy()
base_preclean.to_parquet(PRECLEAN_DIR / "micro_preclean_base.parquet")

#Robustness: full panel (1995-2017)
robust_preclean = panel.copy()
robust_preclean.to_parquet(PRECLEAN_DIR / "micro_preclean_robust.parquet")

print("Preclean base:  ", base_preclean.shape)
print("Preclean robust:", robust_preclean.shape)

## 10. Baseline set, 2004-2017

Drop above 9% missing, impute between 5% and 9%, keep below 5%. `cr_llr` and `x_obsi` are
dropped regardless of their rate because they are not populated consistently enough across
countries to compare.

For the 2026 rerun the selection is **frozen at the published sets**. The sample corrections in
section 4b change the missingness shares, and re-applying the thresholds would add variables that
were never in the thesis models. The rule's choice on the corrected sample is still computed and
saved next to the frozen treatment (`treatment_by_rule`).

In [ ]:
id_cols = ["bank_id","year"]

miss = base_preclean.drop(columns=id_cols).isna().mean()

# The rule agreed with the supervisors: drop above 9% missing, plus cr_llr and x_obsi regardless.
drop_by_rule = miss[miss > 0.09].index.tolist()
drop_by_rule += [c for c in ["cr_llr","x_obsi"] if c in base_preclean.columns]
drop_by_rule = sorted(set(drop_by_rule))

# The 2026 rerun keeps the PUBLISHED selection. Removing the near-empty entities in section 4b lowers
# the missingness shares, and the rule would then admit variables that were never in the thesis
# models (cr_impchg at 4.5%, r_loans_dep at 5.7%). That would change the specification rather than
# fix an error, so the selection is frozen here. RESELECT_VARIABLES = True applies the rule instead.
RESELECT_VARIABLES = False
PUBLISHED_BASE_DROP = ["Tier 1 Ratio", "c_capratio", "cr_impchg", "cr_llr", "cr_npl", "g_assets",
                       "g_loans", "r_ib_ratio", "r_loans_dep", "x_obsi"]

drop_vars = drop_by_rule if RESELECT_VARIABLES else sorted(c for c in PUBLISHED_BASE_DROP if c in base_preclean.columns)

print("Dropping:", drop_vars)
if drop_vars != drop_by_rule:
    print("The rule on the corrected sample would drop:", drop_by_rule)

base_filtered = base_preclean.drop(columns=drop_vars).copy()
base_filtered.shape

In [ ]:
miss2 = base_filtered.drop(columns=id_cols).isna().mean() #recalculate missingness after dropping the worst offenders

impute_by_rule = miss2[(miss2 > 0.05) & (miss2 <= 0.09)].index.tolist()

# Frozen with the selection above: the published imputation set, not the rule on the corrected sample.
PUBLISHED_BASE_IMPUTE = ["c_leverage", "l_loans", "r_cti", "r_dep_liab", "r_loans_assets", "x_dep",
                         "x_ib_liab", "x_loans_gross", "x_loans_net"]
impute_vars = impute_by_rule if RESELECT_VARIABLES else [c for c in PUBLISHED_BASE_IMPUTE if c in base_filtered.columns]
keep_vars = [c for c in miss2.index if c not in impute_vars]

print("Impute:", impute_vars)
if sorted(impute_vars) != sorted(impute_by_rule):
    print("The rule on the corrected sample would impute:", impute_by_rule)
print("Keep:", keep_vars[:10], "...")

In [ ]:
base_final = base_filtered.copy()

if len(impute_vars) > 0:
    imputed_part = panel_impute(base_filtered[id_cols + impute_vars])
    base_final[impute_vars] = imputed_part[impute_vars]

In [ ]:
miss_df = (miss*100).round(3).reset_index()
miss_df.columns = ["variable","missing_pct"]

miss_df.to_csv("../data/micro/clean/missingness_baseline_2004_2017.csv", index=False)

In [ ]:
import pandas as pd

decision_df = pd.DataFrame({
    "variable": miss.index,
    "missing_pct": (miss*100).round(3),
    "treatment": [
        "drop" if v in drop_vars else
        "impute" if v in impute_vars else
        "keep"
        for v in miss.index
    ],
    # what the agreed rule would have chosen on the corrected sample, for transparency (section 10)
    "treatment_by_rule": [
        "drop" if v in drop_by_rule else
        "impute" if v in impute_by_rule else
        "keep"
        for v in miss.index
    ],
})

decision_df.to_csv("../data/micro/clean/baseline_variable_treatment.csv", index=False)
decision_df.head(20)

In [ ]:
treat = pd.read_csv("../data/micro/clean/baseline_variable_treatment.csv")

impute_cols = treat.loc[treat.treatment=="impute", "variable"].tolist()
keep_cols = treat.loc[treat.treatment=="keep", "variable"].tolist()

miss_final = base_final.drop(columns=["bank_id","year"]).isna().mean()

print("Max missing in imputed cols (%):", (miss_final[impute_cols].max()*100) if impute_cols else 0)
print("Max missing in kept cols (%):", (miss_final[keep_cols].max()*100) if keep_cols else 0)

### Selection on data up to 2009 (audit A18, check only)

The thresholds above are computed on the whole window, test years included. This check applies the
same rules to data up to 2009, the last year before the first test year, and prints what would change.
It is not applied: see RERUN.md.

In [ ]:
def rule_selection(df, lo, hi, forced_drop, lo_inclusive):
    m = df.drop(columns=["bank_id", "year"]).isna().mean()
    drop = set(m[m > hi].index) | {c for c in forced_drop if c in m.index}
    imp = set(m[(m >= lo) if lo_inclusive else (m > lo)].index) & set(m[m <= hi].index) - drop
    return {v: "drop" if v in drop else "impute" if v in imp else "keep" for v in m.index}

for label, full, early, lo_incl, hi, forced in [
    ("baseline", base_preclean, base_preclean[base_preclean["year"] <= 2009], False, 0.09, ["cr_llr", "x_obsi"]),
    ("robustness", robust_preclean, robust_preclean[robust_preclean["year"] <= 2009], True, 0.12, []),
]:
    a = rule_selection(full, 0.05, hi, forced, lo_incl)
    b = rule_selection(early, 0.05, hi, forced, lo_incl)
    changes = {v: f"{a[v]} -> {b[v]}" for v in a if a[v] != b[v]}
    print(f"{label}: {len(changes)} variables would change category when selected on data up to 2009")
    for v, ch in changes.items():
        print(f"   {v:18s} {ch}")

## 11. Robustness set, 1995-2017

The same three-way split at a looser threshold, 12% rather than 9%. `x_obsi` is kept here
and log-transformed instead of dropped.

In [ ]:
id_cols = ["bank_id","year"]

miss_r = robust_preclean.drop(columns=id_cols).isna().mean().sort_values(ascending=False)
(miss_r*100).round(2).head(20)

In [ ]:
miss_r = robust_preclean.drop(columns=id_cols).isna().mean()

drop_by_rule_r   = miss_r[miss_r > 0.12].index.tolist() #since this is a sensitivity check, the purpose is to test whether the findings hold under different conditions, being slightly more lenient means keeping more variables
impute_by_rule_r = miss_r[(miss_r >= 0.05) & (miss_r <= 0.12)].index.tolist()

# Frozen at the published selection for the same reason as the baseline set (section 10). On the
# corrected sample the rule would admit r_ib_ratio, cr_llr, g_assets and g_loans.
PUBLISHED_ROBUST_DROP = ["Tier 1 Ratio", "c_capratio", "cr_llr", "cr_npl", "g_assets", "g_loans", "r_ib_ratio"]
PUBLISHED_ROBUST_IMPUTE = ["c_eqratio", "c_leverage", "cr_impchg", "exp_total", "inc_nii", "inc_op",
                           "l_loans", "p_nim", "r_cti", "r_dep_liab", "r_loans_assets", "r_loans_dep",
                           "x_dep", "x_equity", "x_ib_liab", "x_liab", "x_liab_equity", "x_loans_gross",
                           "x_loans_net", "x_obsi"]
if RESELECT_VARIABLES:
    drop_vars_r, impute_vars_r = drop_by_rule_r, impute_by_rule_r
else:
    drop_vars_r   = [c for c in PUBLISHED_ROBUST_DROP if c in robust_preclean.columns]
    impute_vars_r = [c for c in PUBLISHED_ROBUST_IMPUTE if c in robust_preclean.columns]
keep_vars_r = [c for c in miss_r.index if c not in drop_vars_r + impute_vars_r]

print("Drop:", drop_vars_r)
print("Impute:", impute_vars_r[:10])
print("Keep:", keep_vars_r[:10])
if sorted(drop_vars_r) != sorted(drop_by_rule_r) or sorted(impute_vars_r) != sorted(impute_by_rule_r):
    print("The rule on the corrected sample would drop:", sorted(drop_by_rule_r))
    print("                                  and impute:", sorted(impute_by_rule_r))

In [ ]:
robust_filtered = robust_preclean.drop(columns=drop_vars_r).copy()

In [ ]:
robust_final = robust_filtered.copy()

if len(impute_vars_r) > 0:
    imputed_part = panel_impute(robust_filtered[id_cols + impute_vars_r])
    robust_final[impute_vars_r] = imputed_part[impute_vars_r]

In [ ]:
robust_final["l_obsi"] = np.where( #the raw level in millions EUR tracks bank and country size, so untransformed it behaves as a scale proxy rather than a risk signal. Logging it keeps it consistent with l_assets and l_loans
    robust_final["x_obsi"] > 0,
    np.log(robust_final["x_obsi"]),
    np.nan, #zero or negative off-balance-sheet exposure is not economically possible, so treat it as missing rather than as a real low value
)
robust_final = robust_final.drop(columns=["x_obsi"])

print("l_obsi added, x_obsi dropped.")
print(robust_final[["l_obsi"]].describe().round(3))
print(f"Missing l_obsi: {robust_final['l_obsi'].isna().mean():.4f}")

In [ ]:
import pandas as pd

decision_r = pd.DataFrame({
    "variable": miss_r.index,
    "missing_pct": (miss_r*100).round(3),
    "treatment": [
        "drop" if v in drop_vars_r else
        "impute" if v in impute_vars_r else
        "keep"
        for v in miss_r.index
    ],
    "treatment_by_rule": [
        "drop" if v in drop_by_rule_r else
        "impute" if v in impute_by_rule_r else
        "keep"
        for v in miss_r.index
    ],
})

decision_r.to_csv("../data/micro/clean/robust_variable_treatment.csv", index=False)

## 12. Country codes and column order

In [ ]:
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except AttributeError: #pycountry returns None for an unknown code, so .alpha_3 is what raises
        return None

for df in (base_final, robust_final): #ORBIS supplies two-letter codes, the macro sources all use three-letter, so convert here to make the merge in notebook 03 possible
    df["country_iso3"] = df["country_iso"].apply(iso2_to_iso3)

print("Missing ISO3 base:  ", base_final["country_iso3"].isna().mean())
print("Missing ISO3 robust:", robust_final["country_iso3"].isna().mean())

base_final = base_final.drop(columns=["country_iso"]).rename(columns={"country_iso3": "country_iso"})
robust_final = robust_final.drop(columns=["country_iso"]).rename(columns={"country_iso3": "country_iso"})

In [ ]:
def move_country_next_to_bank(df): #purely cosmetic, it puts the two identifiers side by side so the saved files are easier to read
    cols = df.columns.tolist()
    cols.remove("country_iso")
    cols.insert(cols.index("bank_id") + 1, "country_iso")
    return df[cols]

base_final = move_country_next_to_bank(base_final)
robust_final = move_country_next_to_bank(robust_final)

base_final.head()

## 13. Save the clean panels

In [ ]:
base_final.to_parquet(CLEAN_DIR / "micro_clean_base.parquet")
robust_final.to_parquet(CLEAN_DIR / "micro_clean_robust.parquet")

print("Clean base:  ", base_final.shape)
print("Clean robust:", robust_final.shape)